In [ ]:
import os
import torch
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/Singleton/0-1"  # Replace with the actual folder path

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base")

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {
    "non-singleton(0)": 0,
    "Singleton(1)": 1
}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)


from collections import Counter
print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in java_programs:
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    np.vstack(program_embeddings),
    program_labels,
    program_files,
    test_size=0.2,
    random_state=0,
    stratify=program_labels
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KeyboardInterrupt: 